# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## Clone Repo and Setup Project root

In [1]:
import os
from pathlib import Path

# Clone repo if not already present
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
    print("✅ Clone complete")
else:
    print("✅ Repo already present — pulling latest...")
    os.system('git -C /content/FIUBench_Reproducing pull')

# Try Colab Drive mount (optional, for saving checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    in_colab = True
except ImportError:
    in_colab = False

# Resolve PROJECT_ROOT
colab_root = '/content/FIUBench_Reproducing'
local_root = '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/fiubench_reproducibility'
PROJECT_ROOT = colab_root if os.path.exists(colab_root) else local_root

assert os.path.exists(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ In Colab: {in_colab}")


✅ Repo already present — pulling latest...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ In Colab: True


## GPU Check

In [2]:
import torch

print("\n" + "="*60)
print("GPU STATUS")
print("="*60)

if torch.cuda.is_available():
    print(f"✅ CUDA available")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA version: {torch.version.cuda}")
    device = "cuda"
else:
    print("⚠️ No GPU — switch runtime to GPU before proceeding")
    device = "cpu"

print(f"✅ PyTorch: {torch.__version__}")
print("="*60 + "\n")



GPU STATUS
✅ CUDA available
   GPU: NVIDIA A100-SXM4-80GB
   VRAM: 85.1 GB
   CUDA version: 12.1
✅ PyTorch: 2.4.1+cu121



## Install Dependencies

In [3]:
import subprocess
import sys
import torch
import transformers

print("Installing dependencies...")

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

print("✅ Dependencies installed")
print(f"✅ torch={torch.__version__}")
print(f"✅ transformers={transformers.__version__}")


Installing dependencies...
✅ Dependencies installed
✅ torch=2.4.1+cu121
✅ transformers=4.48.0


## Load Model + Processor

In [ ]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip uninstall -y torchvision", shell=True)
subprocess.run(f"{sys.executable} -m pip install --no-cache-dir torchvision==0.19.1", shell=True)
print("✅ torchvision reinstalled")

In [ ]:
import subprocess, sys, os
from huggingface_hub import snapshot_download

# 1. Enable the ultra-fast Rust transfer library
subprocess.run(f"{sys.executable} -m pip install -q hf_transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = ""

# 2. Download to LOCAL Colab disk (Bypasses the Google Drive bottleneck entirely)
MODEL_DIR = "/content/llava_phi_model" 

print("Downloading model to local NVMe at maximum speed...")
snapshot_download(
    repo_id="xtuner/llava-phi-3-mini-hf",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    token=os.environ["HF_TOKEN"],
)
print("✅ Done! You can now load the model.")

## Unified Evaluation Harness

All metrics derived directly from `FIUBench/evaluate_util.py`:

- **Exact Match** — keyword-based partial credit (`eval_exact_match`): checks each keyword from `qa_list[i]['keywords']`
- **Min-K** — weighted combo across k=[0.1…0.5] with weights [0.3,0.3,0.2,0.1,0.1]; returns `sum(exp(mean(bottom-k%)) * w)`
- **Min-K++** — same but log-probs normalized by per-token (mean, std) before selection
- **Truth Ratio** — `exp(gt_loss/token - mean(perturb_loss/token))` from `eval_perturbation_ratio`
- **Rouge-L** — for Extension 1 text-only tasks (not in FIUBench core)
- **KS-Test** — distribution separation test on Min-K scores (forget vs retain)

In [ ]:
import re
import math
import json as _json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from rouge_score import rouge_scorer
from scipy import stats
import torch.nn.functional as F

# ── Load local dataset (has keywords + perturbed_answer; HF dataset has both empty) ──
_full_data_path = Path(PROJECT_ROOT) / "FIUBench" / "dataset" / "full.json"
with open(_full_data_path) as _f:
    FULL_DATA = [_json.loads(line) for line in _f if line.strip()]

# Build lookup by unique_id for fast access
FULL_DATA_BY_ID = {row["unique_id"]: row for row in FULL_DATA}
print(f"✅ Loaded local full.json: {len(FULL_DATA)} rows")

# ── helpers ──────────────────────────────────────────────────────────────────

SFHQ_DIR = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"

def resolve_image(image_path_str: str) -> Path:
    filename = Path(image_path_str).name
    return SFHQ_DIR / filename

# ── Rouge-L ───────────────────────────────────────────────────────────────────
# Used for Extension 1 text-only tasks (not in FIUBench core eval)

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def rouge_l(prediction: str, reference: str) -> float:
    return _rouge.score(reference, prediction)["rougeL"].fmeasure

# ── Exact Match (keyword-based, matches evaluate_util.eval_exact_match) ───────
# FIUBench EM checks each keyword from qa_list[i]['keywords'], giving partial credit.
# score = sum(1/n_keywords for each keyword present in prediction)

def exact_match(prediction: str, keywords: list) -> float:
    """keywords: list of strings from qa_list[i]['keywords'] (use local full.json)"""
    if not keywords:
        return 0.0
    score = 0.0
    for key in keywords:
        if key.lower() in prediction.lower():
            score += 1.0 / len(keywords)
    return min(1.0, score)

# ── Min-K and Min-K++ (matches evaluate_util.py exactly) ─────────────────────
# Weighted combo across 5 k-ratios, weights [0.3, 0.3, 0.2, 0.1, 0.1]
# Min-K  : sum(exp(mean(bottom-k%)) * w)
# Min-K++: same but log-probs normalized by per-token (mean, std) first

MINK_RATIOS  = [0.1, 0.2, 0.3, 0.4, 0.5]
MINK_WEIGHTS = [0.3, 0.3, 0.2, 0.1, 0.1]

@torch.no_grad()
def _get_answer_token_logprobs(question: str, answer: str, image_path: str):
    """
    Multimodal forward pass; returns (token_log_probs, mu, sigma) over answer tokens.
    token_log_probs: numpy (A,)
    mu, sigma      : per-token mean/std of log-prob distribution (for Min-K++)
    """
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return None, None, None
    img = Image.open(img_path).convert("RGB")

    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer

    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            pixel_values   = inputs["pixel_values"],
        )

    answer_logits = out.logits[0][prompt_len - 1: -1, :]   # (A, V)
    answer_ids    = inputs["input_ids"][0][prompt_len:]     # (A,)

    log_probs = F.log_softmax(answer_logits.float(), dim=-1)   # (A, V)
    probs     = log_probs.exp()                                 # (A, V)

    token_lp = log_probs.gather(-1, answer_ids.unsqueeze(-1)).squeeze(-1)  # (A,)
    mu       = (probs * log_probs).sum(-1)                                  # (A,)
    sigma    = ((probs * log_probs.pow(2)).sum(-1) - mu.pow(2)).clamp(min=1e-8).sqrt()  # (A,)

    result = (token_lp.cpu().numpy(), mu.cpu().numpy(), sigma.cpu().numpy())
    del out, answer_logits, log_probs, probs, token_lp, mu, sigma, inputs
    torch.cuda.empty_cache()
    return result


def mink(question: str, answer: str, image_path: str) -> float:
    """Weighted Min-K. Matches evaluate_util.py lines 458-465."""
    token_lp, _, _ = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    scores = []
    for ratio in MINK_RATIOS:
        k = max(1, int(len(token_lp) * ratio))
        topk = np.sort(token_lp)[:k]
        scores.append(float(np.exp(np.mean(topk))))
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS) if not math.isnan(s))


def mink_plus_plus(question: str, answer: str, image_path: str) -> float:
    """Weighted Min-K++. Matches evaluate_util.py lines 467-475."""
    token_lp, mu, sigma = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    normalized = (token_lp - mu) / (sigma + 1e-8)
    scores = []
    for ratio in MINK_RATIOS:
        k = max(1, int(len(normalized) * ratio))
        topk = np.sort(normalized)[:k]
        scores.append(float(np.exp(np.mean(topk))))
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS) if not math.isnan(s))


# ── Truth Ratio (matches eval_perturbation_ratio in evaluate_util.py) ────────
# truth_ratio = exp(gt_loss/token - mean(perturb_loss/token))
# > 1: model ranks gt above perturbed (knowledge retained)
# After forgetting, forget-set truth_ratio should drop toward/below 1

@torch.no_grad()
def _answer_loss_per_token(question: str, answer: str, image_path: str) -> float:
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return float("nan")
    img = Image.open(img_path).convert("RGB")

    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer

    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100  # mask prompt

    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            pixel_values   = inputs["pixel_values"],
            labels         = labels,
        )

    loss = out.loss.item()
    del out, inputs, labels
    torch.cuda.empty_cache()
    return loss


def truth_ratio(question: str, answer: str, perturbed_answers: list, image_path: str) -> float:
    """perturbed_answers: qa_list[i]['perturbed_answer'] from local full.json"""
    if not perturbed_answers:
        return float("nan")
    gt_loss = _answer_loss_per_token(question, answer, image_path)
    if math.isnan(gt_loss):
        return float("nan")
    perturb_losses = [_answer_loss_per_token(question, pa, image_path) for pa in perturbed_answers]
    perturb_losses = [pl for pl in perturb_losses if not math.isnan(pl)]
    if not perturb_losses:
        return float("nan")
    return math.exp(gt_loss - float(np.mean(perturb_losses)))


# ── KS-Test ───────────────────────────────────────────────────────────────────

def ks_test(forget_scores: list, retain_scores: list) -> dict:
    f = [s for s in forget_scores if not math.isnan(s)]
    r = [s for s in retain_scores if not math.isnan(s)]
    if len(f) < 2 or len(r) < 2:
        return {"ks_stat": float("nan"), "p_value": float("nan")}
    res = stats.ks_2samp(f, r)
    return {"ks_stat": res.statistic, "p_value": res.pvalue}


# ── Smoke test ────────────────────────────────────────────────────────────────

print("Running smoke test (using local full.json)...")
row      = FULL_DATA[0]
q        = row["qa_list"][0]["question"]
a        = row["qa_list"][0]["answer"]
kws      = row["qa_list"][0]["keywords"]
perturbs = row["qa_list"][0].get("perturbed_answer", [])
img_p    = row["image_path"]

print(f"  Q: {q}")
print(f"  A: {a[:70]}...")
print(f"  keywords: {kws}")
print(f"  # perturbed answers: {len(perturbs)}")

# EM — gold answer vs its own keywords (should be 1.0 if keywords appear in answer)
em = exact_match(a, kws)
print(f"  EM (gold vs keywords): {em:.3f}  {'✅' if em > 0 or not kws else '⚠️ keywords not in answer'}")

# Rouge-L gold vs gold — must be 1.0
print(f"  Rouge-L (gold vs gold): {rouge_l(a, a):.3f}")

# Min-K and Min-K++ — pretrained never saw these identities, expect very small (~1e-4 to 1e-7)
mk  = mink(q, a, img_p)
mk2 = mink_plus_plus(q, a, img_p)
print(f"  Min-K:   {mk:.3e}  (pretrained baseline; paper reports ~0.034 POST Stage-1 fine-tuning)")
print(f"  Min-K++: {mk2:.3e}")

# Truth ratio — pretrained should have < 1 (never fine-tuned on this data)
tr = truth_ratio(q, a, perturbs, img_p)
print(f"  Truth ratio: {tr:.4f}  {'(expect < 1 before fine-tuning)' if not math.isnan(tr) else '(skipped)'}")

print("\n✅ Eval harness smoke test passed")


## Download SHFQ Images

In [ ]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dir.mkdir(parents=True, exist_ok=True)

existing = list(sfhq_dir.glob("*.jpg"))
if len(existing) >= 400:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    print("Downloading SFHQ.zip from Hugging Face...")
    zip_path = hf_hub_download(
        repo_id="gray311/FIUBench",
        filename="SFHQ.zip",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Downloaded to: {zip_path}")

    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFHQ.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("Extraction complete")

    found = list(extract_dir.rglob("*.jpg"))
    print(f"Found {len(found)} jpg files in zip")
    for f in found[:3]:
        print(f"  {f.relative_to(extract_dir)}")

    copied = 0
    for src in found:
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    total = len(list(sfhq_dir.glob("*.jpg")))
    print(f"✅ Copied {copied} new images — {total} total in {sfhq_dir}")

## Stage 1 - Fine-tuning

### Run FIUBench finetune.py — Table 7 hyperparameters
lr=2e-5, seed=0, 10 epochs, batch=8, grad_accum=16 (effective batch=128), LoRA r=0 (full fine-tuning of LM + projector, vision tower frozen), data=full.json, 400 identities × 20 QA = 8,000 pairs

In [ ]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Patch finetune.py ─────────────────────────────────────────────────────────
ft_py = FIUBENCH_DIR / "finetune.py"
src = ft_py.read_text()
patched = src

patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

if patched != src:
    ft_py.write_text(patched)
    print("✅ Patched finetune.py: sdpa + bf16 + mixed_precision=bf16")
else:
    print("✅ finetune.py already patched")

content = ft_py.read_text()
assert 'torch_dtype=torch.bfloat16' in content, "FAILED: bfloat16 patch missing"
assert 'mixed_precision="bf16"' in content, "FAILED: mixed_precision patch missing"
assert 'torch_dtype=torch.float16' not in content, "FAILED: float16 still present"
print("✅ Verified: bfloat16 + mixed_precision=bf16 active in finetune.py")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py: fixed image token count check")
    else:
        print("✅ modeling_llava.py already patched or check not found")

# ── Patch finetune.py: fix end_training() called before final save ──────────────
# accelerator.end_training() destroys the distributed process group;
# wait_for_everyone() then fails with 'process group not initialized'.
# Fix: move end_training() to after the save block.
patched = ft_py.read_text()
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
ft_py.write_text(patched)
assert 'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()' in ft_py.read_text(), \
    'FAILED: end_training patch missing'
print('\u2705 Patched finetune.py: end_training() moved after final save')

# ── Symlink SFHQ ──────────────────────────────────────────────────────────────
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = FIUBENCH_DIR / "dataset" / "SFHQ"
if not sfhq_dst.exists():
    sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
    sfhq_dst.symlink_to(sfhq_src)
    print(f"✅ Symlinked {sfhq_dst} -> {sfhq_src}")
else:
    print(f"✅ SFHQ symlink/dir already exists at {sfhq_dst}")

# ── Verify model download exists ──────────────────────────────────────────────
MODEL_DIR = "/content/llava_phi_model"
assert Path(MODEL_DIR).exists(), f"Model not found at {MODEL_DIR} — run the download cell first"
model_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
print(f"✅ Model at {MODEL_DIR}: {model_gb:.1f} GB")

# ── Disk usage check ──────────────────────────────────────────────────────────
import subprocess as _sp
_du = _sp.run("df -h /content", shell=True, capture_output=True, text=True)
print(_du.stdout.strip())

# ── Launch training ───────────────────────────────────────────────────────────
# batch_size=8, gradient_accumulation_steps=16 → effective batch 128, matching paper Table 7.
# save_steps=2310: saves once at midpoint (epoch 5) + final — avoids disk overflow.
LOCAL_CKPT = "/content/stage1_checkpoints"
DRIVE_CKPT = "/content/drive/MyDrive/fiubench_checkpoints/stage1"
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

import subprocess, sys, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29503 finetune.py "
    f"--config-name finetune_llava_phi "
    f"model_id={MODEL_DIR} "
    f"save_dir={LOCAL_CKPT} "
    f"save_steps=2310 "
    f"seed=0 lr=2e-5 "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"2>&1 | tee /tmp/ft_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Training time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Copying checkpoint to Drive...")
    os.system(f"rsync -ah --progress {LOCAL_CKPT}/ {DRIVE_CKPT}/")
    os.system(f"cp /tmp/ft_log.txt {DRIVE_CKPT}/training_log.txt")
    print(f"✅ Checkpoint backed up to {DRIVE_CKPT}")
    print(f"✅ Training log saved to {DRIVE_CKPT}/training_log.txt")
else:
    print(f"❌ Training failed (exit {ret}). Check /tmp/ft_log.txt")
